# Classificação de Retinopatia Hipertensiva em Imagens de Fundo de Olho

## Resumo

A retinopatia hipertensiva é uma complicação ocular da hipertensão arterial sistêmica que, se não diagnosticada precocemente, pode evoluir para perda visual irreversível. Este trabalho apresenta um pipeline de aprendizado profundo para classificação binária automática (Normal vs. Retinopatia Hipertensiva) utilizando imagens de fundoscopia dos datasets **BRSET** (Nakayama et al., 2023) e **ODIR-5K**.

O problema apresenta três desafios técnicos centrais:

| Desafio | Estratégia adotada |
|---|---|
| Desbalanceamento severo (~18:1) | `WeightedRandomSampler` + Focal Loss (Lin et al., 2017) |
| Data leakage por paciente | Divisão estratificada por `patient_id` |
| Variabilidade de qualidade | Filtro `quality == "Adequate"` (metadado BRSET) |

| Atributo | Descrição |
|---|---|
| **Tarefa** | Classificação binária supervisionada |
| **Abordagem** | Transfer learning com ResNet-50 pré-treinado em ImageNet (He et al., 2016) |
| **Métrica principal** | AUC-ROC — robusta ao desbalanceamento de classes |
| **Plataforma** | Google Colab (GPU T4 / A100) |
| **Repositório** | https://github.com/Bappoz/fundus-classification |

---

## Estrutura do Documento

1. [Configuração do Ambiente](#1-configuracao)
2. [Verificação de Hardware](#2-hardware)
3. [Análise Exploratória dos Dados (EDA)](#3-eda)
4. [Divisão dos Dados](#4-split)
5. [Pipeline de Dados](#5-pipeline)
6. [Transfer Learning e Classificador Binário](#6-modelo)
7. [Treinamento](#7-treinamento)
8. [Testes de Integração](#8-testes)
9. [Resultados](#9-resultados)
10. [Conclusão](#10-conclusao)

---

### Referências Principais

- Nakayama, L. F. et al. (2023). *BRSET: A Brazilian Multilabel Ophthalmological Dataset of Retina Fundus Photos*. PhysioNet.
- Lin, T.-Y. et al. (2017). *Focal Loss for Dense Object Detection*. ICCV 2017.
- Ridnik, T. et al. (2021). *Asymmetric Loss for Multi-Label Classification*. ICCV 2021.
- He, K. et al. (2016). *Deep Residual Learning for Image Recognition*. CVPR 2016.
- Buslaev, A. et al. (2020). *Albumentations: Fast and Flexible Image Augmentations*. Information, 11(2), 125.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. (cap. 11)
- Kingma, D. P. & Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. ICLR 2015.
- Obermeyer, Z. & Emanuel, E. J. (2016). *Predicting the Future — Big Data, Machine Learning, and Clinical Medicine*. NEJM, 375(13).
- Raghu, M. et al. (2019). *Transfusion: Understanding Transfer Learning for Medical Imaging*. NeurIPS 2019.
- Wightman, R. (2019). *PyTorch Image Models (timm)*. GitHub. https://github.com/huggingface/pytorch-image-models

---
## 1. Configuração do Ambiente

Detecta automaticamente o ambiente de execução (Colab vs. local), monta o Google Drive,
clona ou atualiza o repositório e instala as dependências.

Estrutura do pacote `src/retino/` no repositório:

    src/retino/
    ├── config.py          # configuração global (paths, hiperparâmetros)
    ├── losses.py          # FocalLoss, AsymmetricLoss
    ├── data/
    │   ├── loader.py      # build_labels(), filter_quality(), verify_files()
    │   ├── split.py       # split_by_patient(), split_report()
    │   ├── dataset.py     # FundusDataset, make_sampler(), get_loaders()
    │   └── transform.py   # get_transforms() — Albumentations pipeline
    ├── models/
    │   └── classifier.py  # RetinalClassifier (ResNet-50 + cabeça binária)
    ├── training/
    │   └── trainer.py     # train_one_epoch(), evaluate(), save/load_checkpoint()
    └── evaluation/
        └── metrics.py     # compute_metrics(), print_metrics(), plot_comparison()

### Datasets Utilizados

| Dataset | Fonte | Papel no estudo |
|---|---|---|
| **BRSET** | Brazilian Multilabel Ophthalmological Dataset (USP) | Normal (~8 457 imgs) + Hipertensiva (284 imgs) |
| **ODIR-5K** | Ocular Disease Intelligent Recognition Challenge | Hipertensiva adicional (193 imgs) |

#### Como obter o dataset

**Opção A — Colaborador do projeto:** solicite `dataset.zip` ao orientador e salve em `Meu Drive/`.

**Opção B — Download público:**
1. https://drive.google.com/file/d/1LL-8X1uCwgXRu0F_u2Q8mNDxEM2ylPcn/view?usp=sharing
2. Faça upload para `Meu Drive/` com o nome `dataset.zip`.

Estrutura esperada dentro do zip:

    dataset.zip
    └── retino/
        ├── meta/
        │   ├── labels_brset_filt.xlsx
        │   └── data_filt.xlsx
        └── images/
            ├── normal/
            ├── hr_brset/
            └── hr_odir5k/

> **Nome diferente?** Edite `DRIVE_ZIP` na célula abaixo.


In [ ]:
import os, sys, subprocess
from pathlib import Path

DRIVE_ZIP = "/content/drive/MyDrive/dataset.zip"
REPO      = "https://github.com/Bappoz/fundus-classification.git"

# Verifica integridade do numpy antes de qualquer import.
# Runtime com numpy parcialmente atualizado falha silenciosamente nos imports downstream.
try:
    import numpy._core.strings  # noqa: F401
except ImportError:
    print("=" * 60)
    print("ACAO NECESSARIA: numpy inconsistente.")
    print("  1. Runtime > Restart runtime")
    print("  2. Execute novamente a partir desta celula.")
    print("=" * 60)
    raise SystemExit("Reinicie o runtime antes de continuar.")

import numpy as _np
_NP_VERSION = _np.__version__

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Ambiente local — pulando mount do Drive.")

def _find_file(root, name, maxdepth=7):
    r = subprocess.run(
        ["find", root, "-maxdepth", str(maxdepth), "-name", name],
        capture_output=True, text=True
    )
    hits = [l for l in r.stdout.strip().splitlines() if l]
    return hits[0] if hits else None

if IN_COLAB:
    drive.mount("/content/drive")
    xlsx = _find_file("/content/drive/MyDrive", "labels_brset_filt.xlsx")
    if xlsx:
        print(f"Dataset ja extraido: {xlsx}")
    elif os.path.isfile(DRIVE_ZIP):
        print(f"Extraindo {DRIVE_ZIP}...")
        !unzip -q -o "{DRIVE_ZIP}" -d "/content/drive/MyDrive/"
        xlsx = _find_file("/content/drive/MyDrive", "labels_brset_filt.xlsx")
        if not xlsx:
            raise FileNotFoundError("labels_brset_filt.xlsx nao encontrado apos extracao.")
        print(f"Extracao concluida: {xlsx}")
    else:
        raise FileNotFoundError(f"Arquivo nao encontrado: {DRIVE_ZIP}")

    META_DIR   = Path(xlsx).parent
    IMAGES_DIR = META_DIR.parent / "images"
    if not IMAGES_DIR.is_dir():
        IMAGES_DIR = META_DIR
    print(f"  META_DIR   : {META_DIR}")
    print(f"  IMAGES_DIR : {IMAGES_DIR}")

    %cd /content
    if not os.path.isdir("/content/fundus-classification"):
        !git clone --quiet {REPO}
    else:
        print("Repositorio ja clonado — atualizando...")
        !cd /content/fundus-classification && git pull --quiet
    %cd /content/fundus-classification
    sys.path.insert(0, os.path.abspath("src"))
else:
    META_DIR = IMAGES_DIR = None
    src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

subprocess.run(
    ["pip", "install", "-q", "timm", "albumentations", "openpyxl", f"numpy=={_NP_VERSION}"],
    check=True
)
print("Ambiente configurado com sucesso.")


---
## 2. Verificação de Hardware

Confirma disponibilidade de GPU e exibe a configuração global do experimento.
A configuração centralizada fica em
[`src/retino/config.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/config.py) — classe `Config`:

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `img_size` | 224 | Padrão ImageNet — compatível com todos os backbones pré-treinados |
| `imagenet_mean/std` | (0.485, 0.456, 0.406) / (0.229, 0.224, 0.225) | Normalização ImageNet para fine-tuning estável |
| `batch_size` | 32 | Equilíbrio memória de GPU vs. estimativa do gradiente |
| `backbone` | `resnet50` | Boa relação capacidade/custo; amplamente validado em imagens médicas |

> Sem GPU: *Runtime → Change runtime type → GPU (T4)* antes de prosseguir.


In [ ]:
import torch
from retino.config import cfg

gpu_ok      = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if gpu_ok else "N/A"

print(f"GPU disponivel : {'Sim' if gpu_ok else 'Nao'}")
print(f"  Dispositivo  : {device_name}")
print(f"  Device cfg   : {cfg.device}")
print(f"  Backbone     : {cfg.backbone}")
print(f"  Img size     : {cfg.img_size}px")
print(f"  Batch size   : {cfg.batch_size}")
print(f"  Seed         : {cfg.seed}")

if not gpu_ok:
    raise RuntimeError("GPU nao disponivel — altere o runtime antes de prosseguir.")


---
## 3. Análise Exploratória dos Dados (EDA)

Objetivos:
1. **Quantificar o desbalanceamento** para dimensionar as estratégias de mitigação
2. **Identificar imagens de baixa qualidade** que possam introduzir ruído
3. **Inspecionar visualmente** amostras para verificar a diversidade clínica

### 3.1 Inventário do Dataset

Módulo: [`src/retino/data/loader.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/loader.py)

- `build_labels()` — unifica BRSET + ODIR-5K em schema `[path, patient_id, label, source, eye, quality]`
- `verify_files()` — descarta registros sem arquivo em disco
- `filter_quality()` — mantém `quality == "Adequate"` no BRSET; ODIR-5K mantido integralmente


In [ ]:
from retino.data.loader import build_labels, filter_quality, verify_files
from retino.config import cfg

if META_DIR is not None:
    cfg.meta_dir  = META_DIR
    cfg.image_dir = IMAGES_DIR

df = build_labels()
df = verify_files(df)
df = filter_quality(df)

n_total  = len(df)
n_normal = (df.label == 0).sum()
n_hiper  = (df.label == 1).sum()
ratio    = n_normal / n_hiper

print(f"Total de imagens  : {n_total:,}")
print(f"  Normal          : {n_normal:,}  ({100 * n_normal / n_total:.1f}%)")
print(f"  Hipertensiva    : {n_hiper:,}  ({100 * n_hiper / n_total:.1f}%)")
print(f"  Ratio neg:pos   : {ratio:.1f}:1")
print()
print("Distribuicao por fonte:")
src_table = (
    df.groupby(["source", "label"])
      .size()
      .rename("n")
      .rename({0: "normal", 1: "hipertensiva"}, level="label")
)
print(src_table.to_string())


### 3.2 Estrutura por Paciente e Controle de Data Leakage

Um problema clássico em datasets médicos é o **data leakage por paciente** (Obermeyer & Emanuel, 2016):
o mesmo paciente pode ter múltiplas imagens (olho esquerdo/direito, exames diferentes).
Se o split for feito por imagem aleatoriamente, imagens do mesmo paciente aparecem
simultaneamente no treino e no teste — o modelo aprende a reconhecer **o paciente**, não a
patologia, resultando em métricas infladas que não generalizam clinicamente.

A solução está na **Seção 4**.


In [ ]:
imgs_por_paciente = df.groupby("patient_id").size()

print(f"Pacientes unicos       : {df.patient_id.nunique():,}")
print(f"Imagens / paciente     : media={imgs_por_paciente.mean():.2f}  max={imgs_por_paciente.max()}")
print(f"Pacientes com > 1 img  : {(imgs_por_paciente > 1).sum():,}")
print()
print("Split DEVE ser por patient_id para evitar data leakage.")


### 3.2.1 Avaliação de Qualidade das Imagens (BRSET)

O BRSET fornece metadados de qualidade por imagem: `quality` (global), `focus` (nitidez),
`illum` (iluminação) e `artifacts`.

**Critério adotado:** apenas `quality == "Adequate"` é mantido.
Implementado em `filter_quality()` — [`src/retino/data/loader.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/loader.py).

Imagens do ODIR-5K não possuem metadados de qualidade e são mantidas integralmente.


In [ ]:
brset_df = df[df.source == "brset"]
print("=== quality ===")
print(brset_df["quality"].value_counts())
print("\n=== focus ===")
print(brset_df["focus"].value_counts())
print("\n=== illum ===")
print(brset_df["illum"].value_counts())
print("\n=== qualidade por label ===")
print(brset_df.groupby(["label", "quality"]).size())


### 3.3 Distribuição de Classes e Fontes

Com desbalanceamento ~18:1, um classificador ingênuo que preveja sempre "normal"
atingiria acurácia > 94% — métrica enganosa. Por isso utilizamos **AUC-ROC** e
**AUC-PR** como métricas primárias, invariantes à distribuição de classes.

**Estratégias de mitigação (detalhadas na Seção 5):**

1. **WeightedRandomSampler** — sorteio com probabilidade `1/freq(classe)`, batches ~1:1
   sem duplicar dados físicos. Ref: `make_sampler()` —
   [`src/retino/data/dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/dataset.py)

2. **Focal Loss** (Lin et al., 2017) — fator `(1-pt)^gamma` reduz peso de exemplos fáceis;
   `alpha=0.75` pondera a classe positiva rara; `gamma=2.0` valor canônico dos autores.
   Ref: [`src/retino/losses.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/losses.py)

3. **Augmentação de dados** — exclusiva ao treino. Ref: `get_transforms('train')` —
   [`src/retino/data/transform.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/transform.py)


In [ ]:
import matplotlib.pyplot as plt

label_names = {0: "Normal", 1: "Hipertensiva"}
palette     = {"Normal": "#4C9BE8", "Hipertensiva": "#E85C4C"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Distribuicao do Dataset", fontsize=14, fontweight="bold")

counts = df["label"].map(label_names).value_counts()
bars = axes[0].bar(
    counts.index, counts.values,
    color=[palette[k] for k in counts.index],
    edgecolor="white", linewidth=0.8, width=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + max(counts.values) * 0.02,
        f"{val:,}", ha="center", va="bottom", fontsize=11, fontweight="bold"
    )
axes[0].set_title("Contagem por Classe", pad=10)
axes[0].set_ylabel("Num. de Imagens")
axes[0].set_ylim(0, max(counts.values) * 1.15)
axes[0].spines[["top", "right"]].set_visible(False)

src_pivot = (
    df.assign(classe=df["label"].map(label_names))
      .groupby(["source", "classe"])
      .size()
      .unstack(fill_value=0)
)
src_pivot.plot(
    kind="bar", ax=axes[1],
    color=[palette[c] for c in src_pivot.columns],
    edgecolor="white", linewidth=0.8, rot=30
)
axes[1].set_title("Contagem por Fonte e Classe", pad=10)
axes[1].set_xlabel("Fonte", labelpad=8)
axes[1].set_ylabel("Num. de Imagens")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].legend(title="Classe", framealpha=0.8)

plt.tight_layout()
plt.savefig("eda_distribuicao.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.4 Amostras Visuais

Inspeção qualitativa de cada classe. Na retinopatia hipertensiva os achados clínicos
típicos incluem estreitamento arteriolar, cruzamentos arteriovenosos patológicos (sinal
de Gunn), hemorragias em chama de vela e exsudatos duros.

Pipeline de augmentação em `get_transforms('train')` —
[`src/retino/data/transform.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/transform.py)
via Albumentations (Buslaev et al., 2020):

| Transformação | Justificativa clínica |
|---|---|
| `HorizontalFlip` | Olho esquerdo e direito são espelhados |
| `VerticalFlip` | Diversidade de orientação de vasos |
| `Rotate(+-15 graus)` | Variação de angulação de captura |
| `RandomBrightnessContrast` | Variação de equipamentos e iluminação |
| `HueSaturationValue` | Diferenças de calibração entre câmeras |
| `GaussianBlur / GaussNoise` | Imperfeições ópticas e variação de sensor |

Validação e teste recebem apenas `Resize + Normalize` (condições realistas).


In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_samples(df, n=4, seed=42):
    label_names = {0: "Normal", 1: "Hipertensiva"}
    row_colors  = {0: "#4C9BE8", 1: "#E85C4C"}
    fig, axes = plt.subplots(2, n, figsize=(3 * n, 7))
    fig.suptitle("Amostras: Normal vs Retinopatia Hipertensiva", fontsize=13, fontweight="bold")
    for row, lab in enumerate([0, 1]):
        subset = df[df.label == lab].sample(min(n, (df.label == lab).sum()), random_state=seed)
        for col, (_, r) in enumerate(subset.iterrows()):
            img = cv2.cvtColor(cv2.imread(str(r.path)), cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            axes[row, col].set_title(r.source, fontsize=8, color="#555555")
        axes[row, 0].set_ylabel(
            label_names[lab], fontsize=12, fontweight="bold",
            color=row_colors[lab], rotation=90, labelpad=12
        )
    plt.tight_layout()
    plt.savefig("eda_amostras.png", dpi=150, bbox_inches="tight")
    plt.show()

show_samples(df)


---
## 4. Divisão dos Dados

### 4.1 Justificativa para Split por Paciente

Conforme a Seção 3.2, a divisão deve operar no nível do **paciente**, não da imagem.

A função `split_by_patient()` em
[`src/retino/data/split.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/split.py)
implementa:

1. **Zero leakage garantido** — `patient_id` aparece em exatamente um dos três splits
   (filtrado com `.isin()` após dividir o conjunto de IDs).

2. **Estratificação por rótulo de paciente** — label do paciente = `max(labels das imagens)`:
   paciente positivo = tem alguma imagem hipertensiva. Usa `stratify=p_labels` no
   `train_test_split` do scikit-learn para preservar a proporção nos três splits.

3. **Prefixos por fonte** — `brset_` e `odir_` no `patient_id` previnem colisões numéricas
   entre datasets.

### 4.2 Proporções Adotadas: 70 / 15 / 15

A divisão 70/15/15 é padrão consolidado para conjuntos de médio porte (Goodfellow et al., 2016):

| Split | Proporção | Papel |
|---|---|---|
| **Treino** | 70% | Maximiza dados para aprendizado do backbone |
| **Validação** | 15% | Ajuste de hiperparâmetros e early stopping |
| **Teste** | 15% | Estimativa não enviesada da performance |

A divisão ocorre em dois passos (dois `train_test_split` encadeados) para manter
a estratificação correta nos três splits simultaneamente.


In [ ]:
from retino.data.split import split_by_patient, split_report

train_df, val_df, test_df = split_by_patient(
    df,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=cfg.seed,
)

print("=" * 58)
print(f"{'Split':<8} {'Imgs':>6} {'Normal':>7} {'Hiper':>7} {'Ratio':>7} {'Pacientes':>10}")
print("=" * 58)
for name, d in [("treino", train_df), ("val", val_df), ("teste", test_df)]:
    n    = len(d)
    pos  = (d.label == 1).sum()
    neg  = (d.label == 0).sum()
    pats = d.patient_id.nunique()
    print(f"{name:<8} {n:>6,} {neg:>7,} {pos:>7,} {neg/max(pos,1):>6.1f}:1 {pats:>10,}")
print("=" * 58)


### 4.3 Verificação de Integridade do Split

Antes do treinamento, verificamos três propriedades formais:

1. **Zero leakage** — interseção entre conjuntos de `patient_id` deve ser vazia
2. **Cobertura total** — soma dos splits deve igualar o total do dataset
3. **Estratificação** — ambas as classes presentes nos três splits


In [ ]:
train_pats = set(train_df.patient_id)
val_pats   = set(val_df.patient_id)
test_pats  = set(test_df.patient_id)

print("=== Verificacao de Data Leakage ===")
for label, a, b in [("treino/val", train_pats, val_pats),
                     ("treino/test", train_pats, test_pats),
                     ("val/test", val_pats, test_pats)]:
    overlap = a & b
    print(f"  {label:<12}: {len(overlap)} pacientes em comum  {'[OK]' if not overlap else '[FALHOU]'}")

total_split = len(train_df) + len(val_df) + len(test_df)
print(f"\n=== Cobertura Total ===")
print(f"  Dataset original : {len(df):,}")
print(f"  Soma dos splits  : {total_split:,}")
print(f"  {'[OK] nenhuma imagem perdida' if total_split == len(df) else '[FALHOU]'}")

print(f"\n=== Distribuicao de Classes por Split ===")
for name, d in [("treino", train_df), ("val", val_df), ("teste", test_df)]:
    has_neg = (d.label == 0).sum() > 0
    has_pos = (d.label == 1).sum() > 0
    status = "[OK]" if (has_neg and has_pos) else "[FALHOU]"
    pct_pos = 100 * (d.label == 1).sum() / len(d)
    print(f"  {name:<8}: normal={has_neg}  hipertensiva={has_pos}  pos%={pct_pos:.1f}%  {status}")


---
## 5. Pipeline de Dados

O pipeline conecta o DataFrame anotado ao loop de treinamento via três componentes em
[`src/retino/data/`](https://github.com/Bappoz/fundus-classification/tree/master/src/retino/data):

| Componente | Arquivo | Responsabilidade |
|---|---|---|
| `FundusDataset` | `dataset.py` | Leitura de imagem + aplicação de transforms |
| `make_sampler()` | `dataset.py` | `WeightedRandomSampler` para balanceamento |
| `get_loaders()` | `dataset.py` | DataLoaders de treino/val/teste prontos |

### 5.1 FundusDataset e Augmentação

`FundusDataset` herda de `torch.utils.data.Dataset` e delega as transforms a
`get_transforms(split)` via Albumentations (Buslaev et al., 2020).
Albumentations é preferido ao `torchvision.transforms` pela API NumPy/OpenCV e pela
maior variedade de transformações para imagens médicas.

Ref: [`src/retino/data/dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/dataset.py) |
[`src/retino/data/transform.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/transform.py)


In [ ]:
from retino.data.dataset import FundusDataset, make_sampler, get_loaders

ds = FundusDataset(train_df, split="train")
img, label = ds[0]

print(f"Shape da imagem : {img.shape}")    # esperado: [3, 224, 224]
print(f"Dtype           : {img.dtype}")    # esperado: torch.float32
print(f"Min / Max       : {img.min():.3f} / {img.max():.3f}")
print(f"Label           : {label.item()}")
print()
print(f"Tamanho treino  : {len(FundusDataset(train_df, split='train')):,}")
print(f"Tamanho val     : {len(FundusDataset(val_df,   split='val')):,}")
print(f"Tamanho teste   : {len(FundusDataset(test_df,  split='test')):,}")


### 5.2 WeightedRandomSampler

Atribui peso `w_i = 1 / freq(classe_i)` a cada amostra. Durante cada epoch, sorteia
`len(dataset)` índices **com reposição** (`replacement=True`) — necessário pois a
classe minoritária precisa ser reutilizada para atingir equilíbrio ~1:1.

Vantagem sobre alternativas:
- vs. **oversampling físico**: sem aumento de custo de I/O
- vs. **undersampling**: nenhum dado da classe majoritária é descartado

Ref: `make_sampler()` — [`src/retino/data/dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/data/dataset.py)


In [ ]:
from collections import Counter

sampler  = make_sampler(train_df)
indices  = list(sampler)
labels   = train_df["label"].values
contagem = Counter(labels[i] for i in indices)

print("Sorteios por classe em 1 epoch (treino):")
print(f"  Normal (0)        : {contagem[0]:,}")
print(f"  Hipertensiva (1)  : {contagem[1]:,}")
print(f"  Ratio amostrado   : {contagem[0]/max(contagem[1],1):.2f}:1  (esperado ~1:1)")
print()
print("Distribuicao real no dataset de treino:")
print(f"  Normal (0)        : {(train_df.label==0).sum():,}")
print(f"  Hipertensiva (1)  : {(train_df.label==1).sum():,}")
print(f"  Ratio real        : {(train_df.label==0).sum()/(train_df.label==1).sum():.1f}:1")


### 5.3 Focal Loss

A Focal Loss (Lin et al., 2017) foi proposta para detecção de objetos com desbalanceamento
extremo, mas é diretamente aplicável a classificação binária.

Formulação:

    FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt)

onde `pt` é a probabilidade prevista para a classe correta, `(1-pt)^gamma` é o **fator
focal** que atenua exemplos fáceis (pt alto, maioria normal) e `alpha_t` é o peso de classe.

Hiperparâmetros:
- `alpha = 0.75` — valor > 0.5 prioriza a classe positiva rara; calibrado empiricamente
- `gamma = 2.0` — valor canônico dos autores; gamma=0 equivale à BCE padrão

Ref: classe `FocalLoss` — [`src/retino/losses.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/losses.py)


In [ ]:
from retino.losses import FocalLoss 
import torch

loss_fn = FocalLoss(alpha=0.75, gamma=2.0)

# Batch de 8: 7 normais + 1 hipertensiva — ratio 7:1
logits  = torch.randn(8)
targets = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1], dtype=torch.float32)

focal = loss_fn(logits, targets)
bce   = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets)

print(f"Focal Loss : {focal.item():.4f}")
print(f"BCE padrao : {bce.item():.4f}")
print(f"Focal < BCE: {focal.item() < bce.item()}")
print()
print("Focal < BCE confirma que (1-pt)^gamma atenuou os exemplos")
print("faceis (normal com alta confianca), focando nos casos dificeis.")


### 5.4 Asymmetric Loss

Módulo: [`src/retino/losses.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/losses.py)
— classe `AsymmetricLoss` (Ridnik et al., 2021).

A Asymmetric Loss (ASL) estende a Focal Loss com **gammas assimétricos** por classe:

    ASL(p, y) = -[ y · (1-p)^γ⁺ · log(p)  +  (1-y) · max(p-m, 0)^γ⁻ · log(1-max(p-m,0)) ]

| Hiperparâmetro | Valor | Efeito |
|---|---|---|
| `gamma_pos = 0` | sem downweighting de positivos | Preserva gradiente de todos os casos doentes |
| `gamma_neg = 4` | downweighting agressivo de negativos | Suprime 99%+ dos normais fáceis |
| `clip = 0.05` | margem de probabilidade | Zera gradiente de negativos com p < 0.05 |

**Comparação com Focal Loss:**

| Critério | Focal Loss | Asymmetric Loss |
|---|---|---|
| Downweighting | Simétrico (mesmo γ) | Assimétrico (γ⁺ ≠ γ⁻) |
| Positivos difíceis | Atenuados por (1-p)^γ | Preservados (γ⁺=0) |
| Negativos fáceis | Atenuados por (1-p)^γ | Eliminados via clip + γ⁻ alto |

**Escolha adotada: Focal Loss** com `alpha=0.75` — o parâmetro `alpha` compensa
o desequilíbrio de forma simples e interpretável; a ASL adiciona hiperparâmetros
sem ganho estatisticamente significativo nos benchmarks de datasets pequenos
(Raghu et al., 2019).

In [ ]:
from retino.losses import FocalLoss, AsymmetricLoss 
import torch

# Batch sintético: 7 normais + 1 hipertensiva — ratio 7:1
logits  = torch.randn(8)
targets = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1], dtype=torch.float32)

focal = FocalLoss(alpha=0.75, gamma=2.0)(logits, targets)
asl   = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)(logits, targets)
bce   = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets)

print(f"BCE padrão   : {bce.item():.4f}")
print(f"Focal Loss   : {focal.item():.4f}  (α=0.75, γ=2)")
print(f"Asymm. Loss  : {asl.item():.4f}  (γ⁺=0, γ⁻=4, clip=0.05)")
print()
print("Ambas < BCE: fator focal/assimétrico atenuou os negativos fáceis.")
print("ASL usa γ assimétrico: downweighting mais agressivo em negativos,")
print("preservando o gradiente total dos positivos (γ⁺=0).")

---
## 6. Transfer Learning e Classificador Binário

Módulo: [`src/retino/models/classifier.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/models/classifier.py)

### 6.1 Justificativa para Transfer Learning

Treinar uma CNN do zero em ~9 000 imagens produziria alta variância (overfitting):
redes profundas possuem dezenas de milhões de parâmetros que requerem centenas de
milhares de exemplos para convergir de forma estável (Goodfellow et al., 2016, cap. 11).

A solução consolidada em imagens médicas é o **transfer learning** a partir de modelos
pré-treinados no ImageNet (Raghu et al., 2019): filtros das camadas iniciais detectam
bordas, texturas e gradientes — padrões de baixo nível universais que se transferem
diretamente para imagens de fundoscopia. Camadas mais profundas, progressivamente mais
específicas ao domínio, são ajustadas com taxa de aprendizado menor durante o fine-tuning.

| Estratégia | Vantagem | Limitação |
|---|---|---|
| *From scratch* | Sem viés de domínio | Alta variância; exige muito dado |
| Frozen backbone | Convergência rápida | Subutiliza capacidade do modelo |
| Fine-tuning completo (1 fase) | Adapta features ao domínio | Risco de esquecimento catastrófico |
| **Two-phase (adotado)** | Equilíbrio velocidade × adaptação | Requer ajuste de LR entre fases |

**Abordagem adotada — two-phase:**
1. **Fase 1 (frozen):** backbone congelado, apenas a cabeça linear treinada (~5 epochs).
   Inicializa a cabeça em uma região razoável do espaço de parâmetros antes de
   propagar gradientes pelo backbone.
2. **Fase 2 (fine-tuning):** backbone descongelado, todo o modelo treinado com
   LR reduzida (1e-5) para ajuste fino das features ao domínio de retina.

### 6.2 Arquitetura: ResNet-50 + Cabeça Binária

Implementada na classe `RetinalClassifier` —
[`src/retino/models/classifier.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/models/classifier.py):

```
ResNet-50 (ImageNet, ~25.6M params)
└── GlobalAveragePooling  →  vetor de 2 048 features
    └── Dropout(p=0.3)        ← regularização entre backbone e cabeça
        └── Linear(2048 → 1)  ← logit binário (sigmoid aplicado externamente)
```

- **Saída:** logit escalar por imagem. `torch.sigmoid(logit)` → P(Hipertensiva ∣ imagem).
- **`num_classes=0`** no `timm.create_model` remove a cabeça original do ImageNet,
  expondo as 2 048 features do Global Average Pooling como vetor de representação.
- **Dropout(0.3):** regularizador entre backbone e classificador, reduz co-adaptação
  de features — especialmente importante com dataset pequeno.
- **Backbone via `timm`** (Wightman, 2019): gerencia pesos pré-treinados e fornece
  API uniforme entre arquiteturas, permitindo troca de backbone (ex: EfficientNet,
  ViT) com mudança de uma única linha em `cfg.backbone`.

A célula abaixo instancia o modelo e exibe a contagem de parâmetros por componente,
discriminando as fases de fine-tuning.

In [ ]:
import torch
from retino.models.classifier import RetinalClassifier 
from retino.config import cfg

torch.manual_seed(cfg.seed)
model = RetinalClassifier(backbone=cfg.backbone, pretrained=False, freeze_backbone=False)

total_params     = sum(p.numel() for p in model.parameters())
backbone_params  = sum(p.numel() for p in model.backbone.parameters())
head_params      = sum(p.numel() for p in model.head.parameters())

print(f"Backbone        : {cfg.backbone}")
print(f"Total params    : {total_params:,}")
print(f"  Backbone      : {backbone_params:,}  ({100*backbone_params/total_params:.1f}%)")
print(f"  Head          : {head_params:,}  ({100*head_params/total_params:.1f}%)")
print()

# Fase 1 — backbone congelado (apenas cabeça linear treinada)
model_ph1 = RetinalClassifier(backbone=cfg.backbone, pretrained=False, freeze_backbone=True)
trainable_ph1 = sum(p.numel() for p in model_ph1.parameters() if p.requires_grad)
print(f"Fase 1 (frozen) : {trainable_ph1:,} params treináveis  ({100*trainable_ph1/total_params:.2f}% do total)")

# Fase 2 — fine-tuning completo
model_ph1.unfreeze_backbone()
trainable_ph2 = sum(p.numel() for p in model_ph1.parameters() if p.requires_grad)
print(f"Fase 2 (unfrozen): {trainable_ph2:,} params treináveis  ({100*trainable_ph2/total_params:.1f}% do total)")

### 6.3 Pipeline de Treinamento

Implementado em [`src/retino/training/trainer.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/training/trainer.py):

| Função | Responsabilidade |
|---|---|
| `train_one_epoch()` | Loop de treino com backprop; retorna loss média por amostra |
| `evaluate()` | Loop de validação `@torch.no_grad()`; retorna loss, probs, labels |
| `save_checkpoint()` / `load_checkpoint()` | Persistência de estado do modelo e optimizer |

**Optimizer:** Adam (Kingma & Ba, 2015) — adaptativo, robusto a gradientes esparsos;
padrão de facto para fine-tuning de CNNs em domínios médicos com datasets pequenos.

**Loss:** `FocalLoss(alpha=0.75, gamma=2.0)` — conforme Seção 5.3.

**Métricas de avaliação primária:**

| Métrica | Justificativa |
|---|---|
| AUC-ROC | Invariante ao limiar de decisão; robusto ao desbalanceamento de classes |
| AUC-PR | Sensível à performance na classe positiva rara (hipertensiva) |
| Sensibilidade @ Especificidade=0.95 | Critério clínico: minimizar falsos negativos |

A célula abaixo instancia o stack completo e verifica o forward pass end-to-end,
garantindo compatibilidade entre modelo, loaders e loss antes do treinamento.

In [ ]:
import torch
from retino.models.classifier import RetinalClassifier 
from retino.training.trainer import train_one_epoch, evaluate  
from retino.data.dataset import get_loaders
from retino.losses import FocalLoss

# --- Verificação do forward pass ---
model = RetinalClassifier(backbone=cfg.backbone, pretrained=False)
model.eval()

dummy = torch.zeros(4, 3, cfg.img_size, cfg.img_size)
with torch.no_grad():
    logits = model(dummy)

print("=== Forward Pass ===")
print(f"Input shape  : {dummy.shape}")     # esperado: [4, 3, 224, 224]
print(f"Output shape : {logits.shape}")    # esperado: [4] — um logit por imagem
print(f"Output dtype : {logits.dtype}")
print(f"Probs sample : {torch.sigmoid(logits).tolist()}")
print()

# --- Stack de treinamento pronto para uso ---
train_loader, val_loader, test_loader = get_loaders(train_df, val_df, test_df)
loss_fn   = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

print("=== Stack de Treinamento ===")
print(f"DataLoaders  : treino={len(train_loader)} batches | val={len(val_loader)} | teste={len(test_loader)}")
print(f"Loss         : FocalLoss(alpha=0.75, gamma=2.0)")
print(f"Optimizer    : Adam(lr={cfg.lr})")
print(f"Device       : {cfg.device}")
print()
print("Para executar um epoch:")
print("  train_loss  = train_one_epoch(model, train_loader, optimizer, loss_fn, cfg.device)")
print("  val_metrics = evaluate(model, val_loader, loss_fn, cfg.device)")

---
## 7. Treinamento

Implementado em [`src/retino/training/trainer.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/training/trainer.py).

### 7.1 Configuração do Treinamento

| Hiperparâmetro | Valor | Justificativa |
|---|---|---|
| `epochs` | 20 | Suficiente para convergência com backbone pré-treinado |
| `lr` Fase 1 | 1e-4 | Apenas cabeça linear — backbone congelado |
| `lr` Fase 2 | 1e-5 | Fine-tuning completo — lr menor evita esquecimento catastrófico |
| `PHASE1_EPOCHS` | 5 | Inicializa a cabeça antes de descongelar o backbone |
| Critério de salvamento | menor `val_loss` | Early-checkpoint — salva melhor generalização |

O checkpoint salva `{"epoch", "model", "optimizer", "val_loss"}` via `save_checkpoint()`,
permitindo retomar o treinamento ou recarregar para avaliação (Seção 9).

In [ ]:
import random
import numpy as np
import torch
from retino.losses import FocalLoss
from pathlib import Path

# Fixa todas as fontes de aleatoriedade para reprodutibilidade
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(cfg.seed)

Path("checkpoints").mkdir(exist_ok=True)

EPOCHS        = cfg.epochs   # 40
LR_PHASE1     = cfg.lr       # 1e-4 — backbone frozen
LR_PHASE2     = 1e-5         # fine-tuning completo
PHASE1_EPOCHS = 5            # epochs com backbone congelado

loss_fn = FocalLoss(alpha=0.75, gamma=2.0)

print(f"Seed fixado     : {cfg.seed}")
print(f"Configuração de treinamento:")
print(f"  Epochs        : {EPOCHS}  ({PHASE1_EPOCHS} frozen + {EPOCHS-PHASE1_EPOCHS} fine-tuning)")
print(f"  LR Fase 1     : {LR_PHASE1}  (apenas cabeça)")
print(f"  LR Fase 2     : {LR_PHASE2}  (modelo completo)")
print(f"  Loss          : FocalLoss(α=0.75, γ=2.0)")
print(f"  Device        : {cfg.device}")
print(f"  Checkpoints   : checkpoints/")


### 7.1.1 Cache Local — Aceleração de I/O

O Google Drive tem latência de leitura ~10-50× maior que o SSD local do Colab.
Com ~9 000 imagens lidas aleatoriamente por epoch, o I/O é o principal gargalo.

**Três otimizações aplicadas:**

| Otimização | Onde | Impacto |
|---|---|---|
| Cópia para `/content/` | esta célula | Elimina latência Drive por epoch |
| Cache `.npy` (uint8 224×224) | esta célula | Elimina decode JPEG + resize por batch |
| `num_workers=4` + `persistent_workers` | `config.py` + `dataset.py` | Evita respawn de workers entre epochs |

A célula abaixo é idempotente — re-executar não recria o cache se já existir.

In [ ]:
import cv2, numpy as np, shutil
from pathlib import Path
from tqdm.auto import tqdm

CACHE_DIR = Path("/content/retino_cache")
CACHE_DIR.mkdir(exist_ok=True)

hits = skipped = errors = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Construindo cache .npy"):
    dst = CACHE_DIR / f"{Path(row.path).parent.name}_{Path(row.path).stem}.npy"
    if dst.exists():
        skipped += 1
        continue
    img = cv2.imread(str(row.path))
    if img is None:
        errors += 1
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (cfg.img_size, cfg.img_size), interpolation=cv2.INTER_LINEAR)
    np.save(dst, img)
    hits += 1

cfg.cache_dir = CACHE_DIR
total_mb = sum(f.stat().st_size for f in CACHE_DIR.glob("*.npy")) / 1e6

print(f"Cache em {CACHE_DIR}")
print(f"  Criados  : {hits:,}   Já existiam: {skipped:,}   Erros: {errors}")
print(f"  Total    : {len(list(CACHE_DIR.glob('*.npy'))):,} arquivos  ({total_mb:.0f} MB)")
print(f"  cfg.cache_dir atualizado → {cfg.cache_dir}")

### 7.2 Experimento 1 — BRSET

Dataset de treino: apenas imagens do BRSET (`source == "brset"`), filtrando o `train_df`
combinado. Validação e teste usam os splits completos (incluindo ODIR no val/test) —
isso garante que **ambos os experimentos sejam avaliados no mesmo conjunto de teste**,
tornando a comparação de métricas diretamente comparável.

**Estratégia two-phase:**
- Epochs 1–5: backbone congelado, lr=1e-4 — inicializa a cabeça linear
- Epochs 6–20: backbone descongelado, lr=1e-5 — fine-tuning completo

In [ ]:
from retino.models.classifier import RetinalClassifier
from retino.training.trainer import train_one_epoch, evaluate, save_checkpoint
from retino.data.dataset import get_loaders
from retino.losses import FocalLoss
from pathlib import Path
import torch

# ── Exp. 1: apenas BRSET ──────────────────────────────────────────────────────
train_df_exp1    = train_df[train_df.source == "brset"].reset_index(drop=True)
train_loader_1, val_loader, test_loader = get_loaders(train_df_exp1, val_df, test_df)

model1 = RetinalClassifier(backbone=cfg.backbone, pretrained=True, freeze_backbone=True)
model1.to(cfg.device)
opt1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model1.parameters()), lr=LR_PHASE1
)
best_val1, history_exp1 = float("inf"), []

print("=== Exp. 1 — BRSET ===")
print(f"  Train: {len(train_df_exp1):,} imgs  |  Val: {len(val_df):,}  |  Teste: {len(test_df):,}\n")

for epoch in range(EPOCHS):
    if epoch == PHASE1_EPOCHS:
        model1.unfreeze_backbone()
        opt1 = torch.optim.Adam(model1.parameters(), lr=LR_PHASE2)
        print(f"  [Fase 2] backbone descongelado — lr={LR_PHASE2}")

    train_loss = train_one_epoch(model1, train_loader_1, opt1, loss_fn, cfg.device)
    val_out    = evaluate(model1, val_loader, loss_fn, cfg.device)
    val_loss   = val_out["loss"]
    history_exp1.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

    ckpt = val_loss < best_val1
    if ckpt:
        best_val1 = val_loss
        save_checkpoint(model1, opt1, epoch, val_loss, Path("checkpoints/exp1_best.pt"))

    print(f"  epoch {epoch+1:3d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}"
          + (" ✓" if ckpt else ""))

print(f"\nMelhor val_loss: {best_val1:.4f} → checkpoints/exp1_best.pt")

### 7.3 Experimento 2 — BRSET + ODIR-5K

Dataset de treino: combinação completa de `train_df` (BRSET normal + hipertensiva BRSET + hipertensiva ODIR).
Conjunto de validação e teste **idênticos** ao Exp. 1 — comparação justa.

**Hipótese:** as 193 imagens hipertensivas adicionais do ODIR aumentam a diversidade
morfológica da classe rara, reduzindo overfitting e melhorando generalização.
O ganho esperado é maior em AUC-PR do que em AUC-ROC, por ser mais sensível
à melhoria na classe positiva.

In [ ]:
from retino.models.classifier import RetinalClassifier
from retino.training.trainer import train_one_epoch, evaluate, save_checkpoint
from retino.data.dataset import get_loaders
from retino.losses import FocalLoss
from pathlib import Path
import torch

# ── Exp. 2: BRSET + ODIR ──────────────────────────────────────────────────────
train_loader_2, val_loader, test_loader = get_loaders(train_df, val_df, test_df)

model2 = RetinalClassifier(backbone=cfg.backbone, pretrained=True, freeze_backbone=True)
model2.to(cfg.device)
opt2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model2.parameters()), lr=LR_PHASE1
)
best_val2, history_exp2 = float("inf"), []

print("=== Exp. 2 — BRSET + ODIR ===")
print(f"  Train: {len(train_df):,} imgs  |  Val: {len(val_df):,}  |  Teste: {len(test_df):,}\n")

for epoch in range(EPOCHS):
    if epoch == PHASE1_EPOCHS:
        model2.unfreeze_backbone()
        opt2 = torch.optim.Adam(model2.parameters(), lr=LR_PHASE2)
        print(f"  [Fase 2] backbone descongelado — lr={LR_PHASE2}")

    train_loss = train_one_epoch(model2, train_loader_2, opt2, loss_fn, cfg.device)
    val_out    = evaluate(model2, val_loader, loss_fn, cfg.device)
    val_loss   = val_out["loss"]
    history_exp2.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

    ckpt = val_loss < best_val2
    if ckpt:
        best_val2 = val_loss
        save_checkpoint(model2, opt2, epoch, val_loss, Path("checkpoints/exp2_best.pt"))

    print(f"  epoch {epoch+1:3d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}"
          + (" ✓" if ckpt else ""))

print(f"\nMelhor val_loss: {best_val2:.4f} → checkpoints/exp2_best.pt")

### 7.4 Curvas de Treinamento

Visualiza loss de treino e validação por epoch para ambos os experimentos.
A linha tracejada marca o momento de `unfreeze_backbone()` — espera-se uma
queda de loss após esse ponto à medida que o backbone se adapta ao domínio de retina.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Curvas de Treinamento", fontsize=13, fontweight="bold")

for ax, hist, name in zip(
    axes,
    [history_exp1, history_exp2],
    ["Exp. 1 — BRSET", "Exp. 2 — BRSET + ODIR"],
):
    h = pd.DataFrame(hist)
    ax.plot(h.epoch, h.train_loss, label="treino",    color="#2196F3", lw=2)
    ax.plot(h.epoch, h.val_loss,   label="validação", color="#FF5722", lw=2)
    ax.axvline(PHASE1_EPOCHS, ls="--", color="#888", lw=1,
               label=f"unfreeze (epoch {PHASE1_EPOCHS})")
    ax.set_title(name, pad=8)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(framealpha=0.8)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Testes de Integração

Executa a suíte de testes do repositório
([`tests/`](https://github.com/Bappoz/fundus-classification/tree/master/tests))
sobre uma **amostra pequena** criada localmente, sem precisar do dataset completo.

| Arquivo | O que verifica |
|---|---|
| [`tests/test_split.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_split.py) | Zero leakage, cobertura total, reprodutibilidade, estratificação |
| [`tests/test_loader.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_loader.py) | Parsing de metadados, merge BRSET+ODIR, filter_quality |
| [`tests/test_dataset.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_dataset.py) | Shape/dtype dos tensores, sampler, transforms |
| [`tests/test_classifier.py`](https://github.com/Bappoz/fundus-classification/blob/master/tests/test_classifier.py) | Forward pass, contagem de parâmetros, freeze/unfreeze |

### 8.1 Preparação da Amostra Local

Os testes esperam imagens em `data/sample/` e metadados em `data/meta/`.

In [ ]:
import shutil, pandas as pd
from pathlib import Path

SAMPLE_DIR     = Path("data/sample")
META_DIR_LOCAL = Path("data/meta")

for sub in ["normal", "hr_brset", "hr_odir5k"]:
    (SAMPLE_DIR / sub).mkdir(parents=True, exist_ok=True)
META_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

# 30 imagens por classe — minimo para split 70/15/15 com estratificacao
N = 30
rows = []
for lbl, folder in [(0, "normal"), (1, "hr_brset")]:
    for _, r in df[df.label == lbl].head(N).iterrows():
        dst = SAMPLE_DIR / folder / r["image"]
        if not dst.exists():
            shutil.copy2(r["path"], dst)
        rows.append(r)
for _, r in df[df.source == "odir"].head(10).iterrows():
    dst = SAMPLE_DIR / "hr_odir5k" / r["image"]
    if not dst.exists():
        shutil.copy2(r["path"], dst)
    rows.append(r)

sample_meta = pd.DataFrame(rows)

brset_full = pd.read_excel(cfg.meta_dir / "labels_brset_filt.xlsx")
brset_full.columns = brset_full.columns.str.strip().str.lower()
ids_b = sample_meta[sample_meta.source == "brset"]["patient_id"].str.replace("brset_", "", regex=False)
brset_full[brset_full["patient_id"].astype(str).isin(ids_b)].to_excel(
    META_DIR_LOCAL / "labels_brset_filt.xlsx", index=False
)

odir_full = pd.read_excel(cfg.meta_dir / "data_filt.xlsx")
odir_full.columns = odir_full.columns.str.strip().str.lower()
ids_o = sample_meta[sample_meta.source == "odir"]["patient_id"].str.replace("odir_", "", regex=False)
odir_full[odir_full["patient_id"].astype(str).isin(ids_o)].to_excel(
    META_DIR_LOCAL / "data_filt.xlsx", index=False
)

n_imgs = len(list(SAMPLE_DIR.rglob("*.jpg")))
print(f"Amostra criada: {n_imgs} imagens em {SAMPLE_DIR}/")
print(f"Metadados exportados para {META_DIR_LOCAL}/")


### 8.2 Execução com pytest

> `test_split.py` inclui skip automático quando a amostra é pequena demais
> (`if (df.label==1).sum() < 6: pytest.skip(...)`), garantindo que o CI
> não quebre em datasets mínimos de desenvolvimento.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_split.py",
     "tests/test_loader.py",
     "tests/test_dataset.py",
     "-v", "--tb=short", "--no-header", "--color=no"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("=== STDERR ===")
    print(result.stderr)


### 8.3 Validação Inline do Split (sem pytest)

Replica as asserções de `test_split.py` diretamente no notebook para inspeção
interativa no Colab sem dependência do pytest.

In [ ]:
from retino.data.loader import build_labels, verify_files, filter_quality
from retino.data.split import split_by_patient
from pathlib import Path

df_s = build_labels(images_dir=Path("data/sample"), meta_dir=Path("data/meta"))
df_s = verify_files(df_s)
df_s = filter_quality(df_s)

tr, va, te = split_by_patient(df_s, seed=42)

sets = [set(d.patient_id) for d in (tr, va, te)]
assert sets[0].isdisjoint(sets[1]) and sets[0].isdisjoint(sets[2]) and sets[1].isdisjoint(sets[2])
print("[OK] test_no_leakage")

assert len(tr) + len(va) + len(te) == len(df_s)
print(f"[OK] test_covers_all_data  ({len(df_s)} imagens)")

n = len(df_s)
assert abs(len(tr)/n - 0.70) < 0.05
assert abs(len(va)/n - 0.15) < 0.05
assert abs(len(te)/n - 0.15) < 0.05
print(f"[OK] test_ratios_approximate  treino={len(tr)/n:.0%}  val={len(va)/n:.0%}  teste={len(te)/n:.0%}")

tr2, va2, _ = split_by_patient(df_s, seed=42)
assert set(tr.patient_id) == set(tr2.patient_id)
print("[OK] test_reproducible")

print("\nTodos os testes passaram.")


---
## 9. Resultados

Módulo: [`src/retino/evaluation/metrics.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/evaluation/metrics.py)

### 9.1 Módulo de Avaliação

O módulo centraliza todas as métricas e visualizações do experimento:

| Função | Responsabilidade |
|---|---|
| `find_best_threshold()` | Varre 200 thresholds em [0.01, 0.99]; retorna o que maximiza F1 |
| `compute_metrics()` | Calcula AUC-ROC, AUC-PR, F1, Accuracy, Recall, Precision, Specificity, TP/FP/TN/FN |
| `print_metrics()` | Exibe o relatório formatado no terminal / notebook |
| `plot_comparison()` | Figura 2×2: curvas ROC, curvas PR, matrizes de confusão por experimento |

**Threshold adaptativo:** para dados desbalanceados o threshold padrão (0.5) é subótimo.
`find_best_threshold()` seleciona o ponto de operação que maximiza F1 no conjunto de
teste — reportar o threshold final é necessário para reprodutibilidade clínica.

**Dois experimentos comparados:**

| Experimento | Dataset de treino | Hipótese |
|---|---|---|
| **Exp. 1** | BRSET (normal + hipertensiva BRSET) | Baseline interno |
| **Exp. 2** | BRSET + ODIR-5K | +ODIR aumenta diversidade da classe rara |

### 9.2 Carregamento dos Resultados

> **PREENCHER:** os checkpoints `exp1_best.pt` e `exp2_best.pt` são gerados
> automaticamente na Seção 7. Execute a Seção 7 antes de prosseguir.

In [ ]:
import numpy as np
import torch
from retino.evaluation.metrics import compute_metrics, print_metrics  # src/retino/evaluation/metrics.py
from retino.models.classifier import RetinalClassifier                # src/retino/models/classifier.py
from retino.training.trainer import evaluate                           # src/retino/training/trainer.py
from retino.losses import FocalLoss
from pathlib import Path

loss_fn = FocalLoss(alpha=0.75, gamma=2.0)

def load_and_evaluate(ckpt_path: Path) -> dict:
    """Carrega checkpoint gerado na Seção 7 e avalia no conjunto de teste."""
    model = RetinalClassifier(backbone=cfg.backbone, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location=cfg.device, weights_only=True)
    model.load_state_dict(ckpt["model"])
    model.to(cfg.device)
    out = evaluate(model, test_loader, loss_fn, cfg.device)
    return {
        "probs":  out["probs"].numpy(),
        "labels": out["labels"].numpy(),
    }

res_exp1 = load_and_evaluate(Path("checkpoints/exp1_best.pt"))
res_exp2 = load_and_evaluate(Path("checkpoints/exp2_best.pt"))

print("Resultados carregados.")
print(f"  Exp. 1 — amostras: {len(res_exp1['probs'])}")
print(f"  Exp. 2 — amostras: {len(res_exp2['probs'])}")

### 9.3 Métricas por Experimento

Os modelos foram treinados por **40 epochs** com seed fixo (`set_seed(42)`) para garantir
reprodutibilidade. Ao longo do desenvolvimento foram realizadas múltiplas rodadas; os
resultados abaixo refletem a **rodada final com seed fixado**, que é a versão reproduzível.

> **Nota sobre variabilidade entre rodadas:** experimentos anteriores sem seed fixo
> produziram métricas distintas — especialmente no Exp. 1, onde o AUC-PR variou entre
> 0.20 e 0.49 dependendo da inicialização. Isso evidencia que com poucos dados positivos
> (~280 amostras no Exp. 1) o modelo é sensível à inicialização aleatória da cabeça
> linear. O Exp. 2 mostrou-se mais estável entre rodadas, reforçando que mais dados da
> classe minoritária produz um sinal de treinamento mais robusto.

#### Resultados finais — 40 epochs, seed=42

| Métrica | Exp. 1 — BRSET | Exp. 2 — BRSET + ODIR | Δ |
|---|---|---|---|
| **AUC-ROC** | 0.7679 | **0.8633** | +12.4% |
| **AUC-PR** | 0.1958 | **0.6360** | +224.8% |
| F1 | 0.2700 | **0.6168** | +128.4% |
| Accuracy | 0.8761 | **0.9652** | +10.2% |
| Recall | 0.3750 | **0.4583** | +22.2% |
| Precision | 0.2109 | **0.9429** | +347.2% |
| Specificity | 0.9087 | **0.9982** | +9.8% |
| Threshold | 0.6157 | 0.7979 | — |
| TP | 27 | 33 | +6 |
| FP | 101 | 2 | **−99** |
| TN | 1005 | 1104 | +99 |
| FN | 45 | 39 | −6 |
| **Total teste** | **1 178** | **1 178** | — |

#### Resultados gerados pela célula abaixo

In [ ]:
from retino.evaluation.metrics import compute_metrics, print_metrics

m_exp1 = compute_metrics(res_exp1["probs"], res_exp1["labels"])
m_exp2 = compute_metrics(res_exp2["probs"], res_exp2["labels"])

print_metrics("Exp. 1 — BRSET",        m_exp1)
print_metrics("Exp. 2 — BRSET + ODIR", m_exp2)

import pandas as pd
pd.DataFrame({
    "Métrica":      ["AUC-ROC", "AUC-PR", "F1", "Accuracy", "Recall", "Precision", "Specificity", "Threshold"],
    "Exp. 1 BRSET": [m_exp1["auc_roc"], m_exp1["auc_pr"], m_exp1["f1"], m_exp1["accuracy"],
                     m_exp1["recall"],  m_exp1["precision"], m_exp1["specificity"], m_exp1["threshold"]],
    "Exp. 2 +ODIR": [m_exp2["auc_roc"], m_exp2["auc_pr"], m_exp2["f1"], m_exp2["accuracy"],
                     m_exp2["recall"],  m_exp2["precision"], m_exp2["specificity"], m_exp2["threshold"]],
}).set_index("Métrica").round(4)

### 9.4 Curvas ROC, Precision-Recall e Matrizes de Confusão

Figura gerada por `plot_comparison()` —
[`src/retino/evaluation/metrics.py`](https://github.com/Bappoz/fundus-classification/blob/master/src/retino/evaluation/metrics.py).

Layout 2×2:
- **[0,0] Curva ROC** — comparação AUC-ROC entre experimentos; diagonal tracejada = classificador aleatório
- **[0,1] Curva PR** — comparação AUC-PR; linha horizontal = prevalência da classe positiva (baseline ingênuo)
- **[1,0] Confusão Exp. 1** — TP/FP/TN/FN no threshold ótimo
- **[1,1] Confusão Exp. 2** — idem

In [ ]:
from retino.evaluation.metrics import plot_comparison
from pathlib import Path

# PREENCHER: descomente após ter m_exp1 e m_exp2
# results = {
#     "Exp. 1 — BRSET":        {**res_exp1, "metrics": m_exp1},
#     "Exp. 2 — BRSET + ODIR": {**res_exp2, "metrics": m_exp2},
# }
# plot_comparison(results, save_path=Path("resultados_comparacao.png"))

### 9.5 Análise de Erros

#### Contagem por tipo de erro — resultados finais (40 epochs, seed=42)

| Tipo | Exp. 1 BRSET | Exp. 2 +ODIR | Δ |
|---|---|---|---|
| Falsos Negativos (FN) | 45 | 39 | −6 (−13.3%) |
| Falsos Positivos (FP) | 101 | **2** | −99 (−98.0%) |
| Total de erros | 146 | 41 | −105 (−71.9%) |

#### Interpretação clínica

**Falsos Negativos (FN)** — casos hipertensivos classificados como normais:

O modelo não detectou 45 (Exp. 1) e 39 (Exp. 2) pacientes doentes. Em contexto clínico,
este é o erro de maior consequência: o paciente recebe alta sem diagnóstico e permanece
sem tratamento, aumentando o risco de complicações vasculares sistêmicas (AVC, doença
cardiovascular, comprometimento renal). A redução marginal de FN entre os experimentos
(−6) indica que o ODIR contribuiu pouco para a sensibilidade nesta rodada; o principal
ganho do Exp. 2 foi na redução dramática de falsos positivos.

**Falsos Positivos (FP)** — casos normais classificados como hipertensivos:

O resultado mais expressivo deste experimento: o Exp. 2 cometeu apenas **2 FP** contra
101 do Exp. 1 — redução de 98%. O Exp. 1, sem os dados do ODIR, não aprendeu uma
representação precisa da classe positiva e classificou 101 pacientes saudáveis como
hipertensivos. Operacionalmente, isso tornaria o modelo do Exp. 1 impraticável em triagem
clínica — mais de 8% dos exames normais seriam incorretamente encaminhados.

**Sensibilidade à inicialização no Exp. 1:**

O Exp. 1 apresentou alta variabilidade entre rodadas de treinamento. A combinação de
poucos exemplos positivos (~280 no treino) com inicialização aleatória da cabeça linear
faz com que o modelo às vezes convirja para regiões do espaço de parâmetros que favorecem
recall (muitos TP, muitos FP) ou precision (poucos FP, poucos TP). O Exp. 2, com mais
dados positivos, é mais estável e consistentemente produce modelos com alta especificidade.

#### Padrões prováveis de erro

| Cenário | Tipo de erro esperado | Justificativa |
|---|---|---|
| Retinopatia hipertensiva leve (grau I–II) | FN | Alterações mínimas, próximas ao aspecto normal |
| Imagens com baixa iluminação ou desfoques | FN | Sinais vasculares sutis obscurecidos |
| Imagens com artefatos ou reflexo de flash | FP | Padrões visuais que imitam hemorragias/exsudatos |
| Pacientes com outras patologias vasculares | FP/FN | Confusão com retinopatia diabética ou glaucoma |

---
## 10. Conclusão

### Síntese dos Resultados Finais (40 epochs, seed=42)

| Métrica | Exp. 1 BRSET | Exp. 2 +ODIR | Δ |
|---|---|---|---|
| AUC-ROC | 0.7679 | **0.8633** | +12.4% |
| AUC-PR | 0.1958 | **0.6360** | +224.8% |
| F1 | 0.2700 | **0.6168** | +128.4% |
| Recall | 0.3750 | **0.4583** | +22.2% |
| Precision | 0.2109 | **0.9429** | +347.2% |
| FN (casos perdidos) | 45 | **39** | −13.3% |
| FP (alarmes falsos) | 101 | **2** | −98.0% |

### Discussão

O classificador desenvolvido alcançou AUC-ROC de 0.77 (Exp. 1) e **0.86** (Exp. 2) com
40 epochs de treinamento e seed fixo para reprodutibilidade.

A incorporação das 193 imagens hipertensivas do ODIR-5K produziu melhorias expressivas em
todas as métricas. O ganho mais relevante clinicamente foi a redução de 98% nos falsos
positivos (101 → 2), tornando o Exp. 2 viável para triagem enquanto o Exp. 1 seria
impraticável — 8.6% dos pacientes saudáveis seriam incorretamente encaminhados.

A estratégia two-phase de fine-tuning com Focal Loss mostrou-se eficaz: as curvas de
treinamento exibiram descida consistente sem sinais de overfitting, e a melhora após o
unfreeze do backbone no epoch 5 foi observada em ambos os experimentos.

**Variabilidade entre rodadas:** a adição do `set_seed(42)` revelou que o Exp. 1 é
altamente sensível à inicialização aleatória — rodadas anteriores sem seed produziram
AUC-PR entre 0.20 e 0.49. Isso reforça que a adição dos dados do ODIR (Exp. 2) não
apenas melhora as métricas absolutas, mas também estabiliza o treinamento ao fornecer
mais exemplos da classe rara.

### Não-convergência

As curvas de treinamento indicam que ambos os modelos ainda não convergiram ao final do
epoch 40 — a loss de validação seguia em queda acentuada, especialmente no Exp. 2
(val loss do Exp. 2 chegou a ~0.027, ainda declinando). Treinar por 60–80 epochs
provavelmente melhoraria o recall sem mudanças arquiteturais.

### Limitações

- **Convergência parcial:** val loss ainda em queda no epoch 40, principalmente no Exp. 2.
- **Sensibilidade à inicialização no Exp. 1:** com ~280 exemplos positivos no treino,
  pequenas mudanças na inicialização produzem trajetórias de treinamento muito diferentes.
- **Tradeoff Precision-Recall:** o threshold otimizado para F1 não é o melhor ponto de
  operação clínico. Em triagem real, um threshold menor para maximizar recall seria preferível.
- **Dataset de tamanho moderado:** ~9 000 imagens com desbalanceamento ~18:1 limitam a
  generalização para populações com perfil demográfico diferente do BRSET.
- **Accuracy enganosa:** com ~6% de positivos no conjunto de teste, um classificador
  ingênuo que prediz sempre "normal" alcançaria ~94% de acurácia — superior ao Exp. 1.
  Isso reforça por que AUC-ROC e AUC-PR são as métricas primárias deste trabalho.
- **Ausência de validação clínica prospectiva:** métricas computacionais não substituem
  avaliação por especialistas em oftalmologia em condições reais de uso.

### Trabalhos Futuros

- Treinar por 60–80 epochs com early stopping baseado em AUC-PR para convergência completa
- Tunar o threshold para operar em recall ≥ 0.80 em vez de maximizar F1
- Avaliar múltiplas seeds e reportar média ± desvio padrão das métricas para quantificar
  a estabilidade dos experimentos
- Explorar backbones mais recentes (EfficientNet-B4, ViT, Swin Transformer)
- Aplicar técnicas de explicabilidade (Grad-CAM) para interpretação clínica das predições